## Dynamic Programming

#### Download necessary modules

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

#### Predicting hourly demand based on 2018 data

In [17]:
# Load cleaned data (2018)
rideData2018 = pd.read_csv("../data/cleanData/df2_2018(clean_parks_metadate).csv")
rideData2018.head()

,date,wdw_ticket_season,dayofweek,dayofyear,weekofyear,monthofyear,year,season,holiday,wdwticketseason,...,hsfirewks,akprdday,akprddt1,akprddt2,akprddn,akfiren,akshwngt,akshwnt1,akshwnt2,akshwnn
0,2018-01-01,peak,2,0,0,1,2018,CHRISTMAS PEAK,1,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
1,2018-01-02,peak,3,1,0,1,2018,CHRISTMAS,0,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
2,2018-01-03,peak,4,2,0,1,2018,CHRISTMAS,0,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
3,2018-01-04,regular,5,3,0,1,2018,CHRISTMAS,0,regular,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
4,2018-01-05,regular,6,4,0,1,2018,CHRISTMAS,0,regular,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light


In [21]:
# Load waiting times data
waitTimes = pd.read_csv("../data/cleanData/animal_kingdom_df1(touringplans_2018).csv")
waitTimes.head()

,park_date,wait_hour,attraction_name,wait_minutes_posted_avg,attraction_duration,attraction_park,attraction_land,park_open,park_close,park_extra_magic_morning,park_extra_magic_evening,park_ticket_season,park_temperature_average,park_temperature_high,attraction_short_name
0,2018-01-01,8,DINOSAUR,15.000000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
1,2018-01-01,9,DINOSAUR,18.333333,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
2,2018-01-01,10,DINOSAUR,23.750000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
3,2018-01-01,11,DINOSAUR,24.000000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
4,2018-01-01,12,DINOSAUR,31.875000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR


In [18]:
# Load attributes of rides 
rideAttributes = pd.read_csv("../data/cleanData/animal_kingdom_ride_attributes.csv")
rideAttributes.head()

,Ride,Speed,Music,Show,Light,Water,3D
0,Dinosaur,Low,Med,High,Med,Low,Low
1,Expedition Everest,High,Low,Med,Med,Low,Low
2,Avatar: Flight of Passage,Medium,High,High,High,Med,High
3,Kilimanjaro Safaris,Low,Low,High,Low,Med,Low
4,Navi River Journey,Low,High,High,High,High,Low


In [22]:
# Create average wait time per ride
average_wait_times = waitTimes.groupby('ride_name')['wait_time'].mean().reset_index()
average_wait_times.columns = ['ride_name', 'average_wait_time']
average_wait_times.head()

KeyError: 'ride_name'

In [19]:
# Use actual average wait times
rideData2018 = rideData2018[['wait_hour', 'attraction_name', 'wait_minutes_actual_avg', 'attraction_duration']]

# Group by ride & hour
agg = rideData2018.groupby(['wait_hour', 'attraction_name'], as_index=False).mean()

# Build wait_table
wait_table = {}
for _, row in agg.iterrows():
    hour = int(row['wait_hour'])
    ride = row['attraction_name']
    wait = row['wait_minutes_actual_avg']
    if hour not in wait_table:
        wait_table[hour] = {}
    wait_table[hour][ride] = wait

# Build ride_time dict (average duration per ride)
ride_time = rideData2018.groupby('attraction_name')['attraction_duration'].mean().to_dict()

print("Wait table sample:", list(wait_table.items())[:3])
print("Ride times:", list(ride_time.items())[:5])

KeyError: "None of [Index(['wait_hour', 'attraction_name', 'wait_minutes_actual_avg',\n       'attraction_duration'],\n      dtype='object')] are in the [columns]"